# Praktikum 3 - Klasifikasi Naive Bayes dengan Data Multimoda

Pada percobaan ini digunakan data teks SMS (`spam.csv`) dengan label **spam** dan **ham**. Fitur teks diekstraksi menggunakan Bag of Words (`CountVectorizer`), kemudian diklasifikasikan dengan **Multinomial Naive Bayes**.

## Langkah 1 - Load Data

Load data ke dalam DataFrame dan lakukan inspeksi sederhana untuk memastikan apakah diperlukan pra-pengolahan data.

In [1]:
import numpy as np
import pandas as pd

# Spesifikasi encoding diperlukan karena data tidak menggunakan UTF-8
df = pd.read_csv('spam.csv', encoding='latin-1')

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


Terdapat 3 kolom yang tidak bermanfaat untuk proses selanjutnya, sehingga kolom tersebut perlu dibuang. Nama kolom **v1** (label) dan **v2** (teks SMS) juga diubah agar lebih mudah dibaca.

## Langkah 2 - Preprocessing

Tahapan preprocessing meliputi:

1. Drop kolom yang tidak digunakan
2. Ubah nama kolom `v1` (label) dan `v2` (teks SMS)
3. Inspeksi data
4. Encode label
5. Memisahkan fitur dengan label

### Langkah 2a - Drop Kolom

In [2]:
# Drop 3 kolom terakhir dengan fungsi iloc
df = df.drop(df.iloc[:, 2:], axis=1)

# Cek data
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2b - Rename Kolom

Kolom `v1` diubah menjadi `Labels` dan `v2` menjadi `SMS`.

In [3]:
# Ubah nama kolom agar lebih mudah dibaca
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Cek data
df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2c - Inspeksi Data

In [4]:
# Cek jumlah data per kelas
print(df['Labels'].value_counts())
print('\n')

# Cek kelengkapan data
print(df.info())
print('\n')

# Cek statistik deskriptif
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Labels  5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


### Langkah 2d - Encoding Label

Label diubah menjadi angka agar dapat diproses oleh model: `spam = 1` dan `ham = 0`.

In [5]:
# Data untuk label
new_labels = {
    'spam': 1,
    'ham': 0
}

# Encode label
df['Labels'] = df['Labels'].map(new_labels)

# Cek data
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2e - Pisahkan Fitur dengan Label

In [6]:
X = df['SMS'].values
y = df['Labels'].values

## Langkah 3 - Ekstraksi Fitur

Ekstraksi fitur menggunakan konsep Bag of Words dengan `CountVectorizer`. Data dibagi terlebih dahulu, kemudian `CountVectorizer` hanya di-*fit* pada data latih dan di-*transform* pada data uji untuk mencegah **information leakage**.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50
)

# Inisiasi CountVectorizer
bow = CountVectorizer()

# Fitting dan transform X_train dengan CountVectorizer
X_train = bow.fit_transform(X_train)

# Transform X_test
# Hanya transform agar model tidak mengetahui parameter yang digunakan
# CountVectorizer saat fitting pada data latih, sehingga data uji tetap
# menjadi data yang asing bagi model
X_test = bow.transform(X_test)

In [8]:
print(len(bow.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

7727
Dimensi data: (4457, 7727)


## Langkah 4 - Training dan Evaluasi Model

Model yang digunakan adalah Multinomial Naive Bayes, kemudian dievaluasi menggunakan akurasi pada data latih dan data uji.

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Inisiasi MultinomialNB
mnb = MultinomialNB()

# Fit model
mnb.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train = mnb.predict(X_train)

# Evaluasi akurasi data training
acc_train = accuracy_score(y_train, y_pred_train)

# Prediksi dengan data testing
y_pred_test = mnb.predict(X_test)

# Evaluasi akurasi data testing
acc_test = accuracy_score(y_test, y_pred_test)

# Print hasil evaluasi
print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229


## Analisis Hasil

1. Dataset berisi 5.572 SMS dengan distribusi kelas yang tidak seimbang, yaitu sekitar 4.825 **ham** dan 747 **spam**. Kondisi ini perlu diperhatikan karena akurasi saja dapat memberi kesan performa yang lebih baik daripada kenyataan, terutama untuk kelas minoritas (spam).
2. `CountVectorizer` di-*fit* hanya pada data latih, lalu digunakan untuk mentransformasi data uji. Langkah ini penting agar kosakata dan parameter vectorizer tidak dipelajari dari data uji sehingga tidak terjadi **information leakage**.
3. Multinomial Naive Bayes memperoleh akurasi data latih sekitar **0,995** dan data uji sekitar **0,978** (nilai dapat sedikit berbeda tergantung versi library). Selisih yang kecil menunjukkan model tidak mengalami overfitting yang berarti.
4. Untuk evaluasi yang lebih menyeluruh pada data tidak seimbang, sebaiknya tambahkan **precision**, **recall**, dan **F1-score** melalui `classification_report`, terutama untuk memastikan model tidak hanya baik dalam memprediksi kelas **ham**.
5. MultinomialNB cocok untuk fitur hasil counting seperti frekuensi kata. Namun, model ini belum mempertimbangkan urutan kata maupun konteks kalimat.